In [ ]:
from minsearch import Index
from gitsource import GithubRepositoryDataReader, chunk_documents
from llmclient import get_llm_client

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [ ]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [ ]:
print(len(documents))

In [ ]:
def build_index(documents):
    chunks = chunk_documents(documents, size=2000, step=1000)
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(chunks)
    return index





In [ ]:
llm_client = get_llm_client(model="gcp/gemini-2.5-pro")

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()


def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append("Filename: " + doc["filename"])
        lines.append("Content: " + doc["content"])
        lines.append("")

    return "\n".join(lines).strip()


def llm(prompt):
    input_messages = [
        {"role": "developer", "content": INSTRUCTIONS},
        {"role": "user", "content": prompt},
    ]

    return llm_client.complete(
        input_messages,
        max_tokens=500,
        temperature=0.7,
    )


def build_prompt(query, search_results):
    context = build_context(search_results)
    return PROMPT_TEMPLATE.format(question=query, context=context)


def search(query, num_results=5):
    """
    Search the github repository for entries matching the given query.
    """
    index = build_index(documents)
    return index.search(query, num_results=num_results)


def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt)

In [ ]:
print(rag("How does the agentic loop keep calling the model until it stops?").model_dump_json(indent=2))
# index = build_index(documents)
# search_results = index.search("How does the agentic loop keep calling the model until it stops?")
# search_results


In [ ]:
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIChatCompletionsRunner, DisplayingRunnerCallback




In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search)


In [ ]:
agent_tools.get_tools()


In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client.for_toyaikit(),
)

In [ ]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)